# 03 - Baseline model
Stratified train/validation/test split, feature engineering fitted on training data, a logistic-regression baseline, and a prior-rate reference.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from credit_risk.config import load_config
from credit_risk.data.loader import load_dataset, dataset_status_report
cfg = load_config()
print(dataset_status_report(cfg))
# demo=True is an EXPLICIT opt-in: if the real Home Credit files are missing, SYNTHETIC data is generated.
df, DATA_SOURCE = load_dataset(cfg, demo=True)
print("\nDATA SOURCE:", DATA_SOURCE, df.shape)
if DATA_SOURCE != "home_credit":
    print("WARNING: SYNTHETIC data - numbers are only a code-path demonstration, not real results.")

In [ ]:
from credit_risk.features.preprocessing import split_data, build_preprocessor, infer_column_groups
from credit_risk.features.engineering import FeatureEngineer
from credit_risk.models.baseline import make_estimator
from credit_risk.models.evaluate import probability_metrics
X_tr,X_va,X_te,y_tr,y_va,y_te = split_data(df,'TARGET',cfg['split']['test_size'],cfg['split']['val_size'],cfg['project']['seed'])
print(len(X_tr),len(X_va),len(X_te), y_tr.mean().round(4), y_va.mean().round(4), y_te.mean().round(4))

In [ ]:
fe = FeatureEngineer(sensitive_columns=cfg['features']['sensitive_columns']).fit(X_tr)
Xtr, Xva = fe.transform(X_tr), fe.transform(X_va)
g = infer_column_groups(Xtr, 15)
prep = build_preprocessor(g, scale_numeric=True, winsorize=(0.01,0.99))
Mtr, Mva = prep.fit_transform(Xtr, y_tr), prep.transform(Xva)   # statistics from TRAIN only
print(Mtr.shape, 'groups:', len(g.numeric), len(g.low_card), len(g.high_card))

In [ ]:
lr = make_estimator('logistic_regression', {'C':0.1,'max_iter':2000}).fit(Mtr, y_tr)
pv = lr.predict_proba(Mva)[:,1]
base = np.full(len(y_va), y_tr.mean())
print('prior baseline:', {k: round(v,4) for k,v in probability_metrics(y_va, base + np.random.default_rng(0).normal(0,1e-9,len(base))).items() if k in ('roc_auc','brier')})
print('logistic regression (validation):', {k: round(v,4) for k,v in probability_metrics(y_va, pv).items() if k in ('roc_auc','pr_auc','ks','brier')})

In [ ]:
names = prep.get_feature_names_out()
coef = pd.Series(lr.coef_[0], index=names).sort_values()
pd.concat([coef.head(8), coef.tail(8)]).to_frame('standardised coefficient')

Coefficients describe associations in this model, not causal effects.